[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/10_grafana/10_proyecto_grafana.ipynb)

# Proyecto 10 · «Salud del agente»: un dashboard como código con navegación entre señales

| | |
|---|---|
| **Nivel** | 🟡 Intermedio |
| **Duración estimada** | ≈ 1 h 30 min |
| **Servicios de Compose** | **Opcionales**: `core` + `metrics` (`docker compose --profile core --profile metrics up -d`, ≈ 0,45 GB). La rúbrica base valida el JSON/YAML **sin Docker**; con stack se ejecutan además todas las consultas contra Grafana |
| **Prerrequisitos** | Lección `10_grafana` (JSON, `dashkit`, variables, data links, alertas) y `09_metricas` (los nombres de las métricas) |
| **Conexión** | Reconstruyes tú el dashboard que la lección te dio hecho. Reutiliza las métricas del proyecto 09 y los enlaces a trazas (07) y logs (08); es el cuadro de mandos del capstone (22) |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. Tras el último incidente, el on-call dijo: *«abrí cuatro dashboards y dos consolas de consulta antes de entender qué pasaba»*. Tu trabajo: **una sola pantalla** que responda, de arriba abajo, *¿estoy bien?* → *¿dónde se va el tiempo?* → *¿qué petición concreta falló?*, con **saltos** a la traza y a los logs, **provisionada como código** (revisable en un *pull request*) y con **una alerta** de síntoma asociada.

## Datos

Telemetría real de SupportOps (métricas, trazas y logs) que genera el propio notebook con `generate_tickets(…, seed=42)` y `LLM_PROVIDER=mock`; tiempos simulados deterministas. Cadena de datos: **stack local** (opcional) → **modo degradado** (validación estructural del JSON/YAML sin servicios) → datos regenerables por semilla. Sin datasets externos.

## Qué vas a construir (los 7 TODOs)

| TODO | Entregable |
|---|---|
| 1 | `make_variables()`: variables `service` (consulta), `tier` (consulta multi-valor con «All») y `search` (texto) |
| 2 | `symptom_row(d)`: *stats* de **tasa de error** y **p95** con umbrales y *data links* a logs y trazas |
| 3 | `trend_row(d)`: tasa por estado (apilada), percentiles p50/p95/p99 y **mapa de calor con exemplars** |
| 4 | `where_row(d)`: p95 del LLM por modelo, latencia por herramienta y pasos p90 |
| 5 | `why_row(d)`: tabla de **trazas** lentas (con enlace en el `traceID`) y panel de **logs** |
| 6 | `finish(d)`: enlaces de dashboard (a las métricas GenAI y a Explore) |
| 7 | `ALERT_YAML`: una regla de alerta de **tasa de error** aprovisionable, enlazada a tu dashboard y panel |

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución) |
|---|---|---|---|
| R1 | Variables | `service` (query, **no** multi), `tier` (query, multi + All con `allValue` `.*`), `search` (textbox) | 3/3 |
| R2 | Síntomas | *stat* de error (`percentunit`, umbrales 0,05 y 0,10, filtra `$service` y `$tier`, `$__rate_interval`) y *stat* de p95 (`histogram_quantile(0.95` sobre `sum by (le)`, umbrales 4 y 6); cada uno con ≥ 1 *data link* a Explore (logs / trazas) | 2/2 |
| R3 | Tendencia | serie apilada `sum by (supportops_status)`; serie con **3** percentiles; mapa de calor con `exemplar: true` y `format: heatmap` sobre `_bucket` | 3/3 |
| R4 | Dónde | 3 paneles con las consultas pedidas (LLM por modelo, herramientas sum/count, pasos p90) | 3/3 |
| R5 | Por qué | tabla Tempo (`traceql`) con override de `traceID` con `${__value.raw}` y panel de logs Loki con `$service`, `$search` y `detected_level` | 2/2 |
| R6 | Navegación | ≥ 3 *data links* en total y ≥ 2 enlaces de dashboard (uno por *tags*) | ≥ 3 / ≥ 2 |
| R7 | Higiene | `dk.lint()` sin problemas (uid, tags, `description` en cada panel, datasource…) y **determinismo** (dos *builds* → mismo hash) | 0 problemas |
| R8 | Alerta | YAML válido: cadena A (Prometheus `instant`) → B (reduce) → C (umbral `gt` entre 0,05 y 0,20), `for` ≥ 30 s, `severity`, `summary` y `runbook_url`, ventana explícita ≥ 1 m (sin `$__rate_interval`), `__dashboardUid__` y `__panelId__` correctos | 11 comprobaciones |
| L1-L2 | *(con stack)* | Todas las consultas ejecutan sin error contra Grafana; el dashboard se sube y se lee | informativo |

> 📏 Los umbrales salen de la solución de referencia (más abajo); son estructurales y no dependen de tu máquina.

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
if str(ROOT) not in sys.path:   # un editable recién instalado no es visible hasta reiniciar el kernel: lo añadimos a mano
    sys.path.insert(0, str(ROOT))
import importlib; importlib.invalidate_caches()
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import json, math, time, collections, pathlib, hashlib, copy, re
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, Incident, ...
from supportops import viz, sample_data, backends
from supportops.telemetry import setup_telemetry, OTelInstrumentor
from supportops.otel_metrics import *                    # MetricsInstrumentor, run_live, prom_instant, prom_range, tempo_spans, ...
from supportops import dashkit as dk                     # dashboards como código
viz.apply_style()
pd.set_option("display.width", 150); pd.set_option("display.max_colwidth", 100)
SEED = 42
FAST_DEV_RUN = True   # True: tráfico en vivo corto (~110 s). False: más largo

import yaml
STACK = check_stack("core+metrics")
LIVE = not STACK.degraded
print("Modo:", "COMPLETO (con Grafana)" if LIVE else "DEGRADADO (rúbrica estructural sin servicios)")
UID = "proj10-salud"       # uid ESTABLE de TU dashboard (no el del curso: salud-agente)
RATE = "$__rate_interval"
SVC = 'service_name="$service"'; TIER = 'supportops_customer_tier=~"$tier"'

## Paso 0 · Lo que te damos

Las funciones de `dashkit` (`stat`, `timeseries`, `heatmap`, `traces_table`, `logs`, `prom`, `loki`, `link`, `explore_url`, `field_link`, `thresholds`, `var_query`, `var_textbox`…) y estas utilidades para inspeccionar tu JSON:

In [ ]:
def panels(d):  return [p for p in dk.walk_panels(d) if p["type"] != "row"]
def titled(d, pat): return [p for p in panels(d) if re.search(pat, p["title"], re.I)]
def exprs(p):  return " ".join(str(t.get("expr", t.get("query", ""))) for t in p.get("targets", []))
def data_links(p):
    ls = list(p.get("links", [])) + list(p["fieldConfig"]["defaults"].get("links", []))
    for o in p["fieldConfig"].get("overrides", []):
        for pr in o["properties"]:
            if pr["id"] == "links": ls += pr["value"]
    return ls
def steps(p): return [s["value"] for s in p["fieldConfig"]["defaults"].get("thresholds", {}).get("steps", [])]

print("Ejemplo de enlace de datos hacia trazas:", dk.explore_url("tempo", {"queryType": "traceql", "query": '{ resource.service.name = "$service" }'})[:110] + "…")
print("Ejemplo de umbrales:", dk.thresholds((None, dk.GREEN), (0.05, dk.ORANGE), (0.10, dk.RED)))

## Paso 1 · Variables (TODO 1)

Devuelve una lista con **tres** variables:
- `service`: consulta `label_values(supportops_agent_requests_total, service_name)`, **un solo valor** (`multi=False`, `all_=False`), valor actual `supportops-agent`.
- `tier`: consulta `label_values(supportops_agent_requests_total, supportops_customer_tier)`, **multi-valor con «All»** (el `allValue` será `.*`).
- `search`: de tipo *textbox* (para filtrar logs).

<details>
<summary>🪜 Pista 1</summary>

`dk.var_query(nombre, consulta, label=..., multi=..., all_=..., current=...)` y `dk.var_textbox(nombre, label=...)`. Con `all_=True` el helper ya pone `allValue: ".*"`.

</details>

In [ ]:
def make_variables() -> list:
    # TODO 1
    raise NotImplementedError("TODO 1")

## Paso 2 · Fila de síntomas (TODO 2)

Añade a `d` (con `d.add(...)`) dos *stats* (`w=4`):

1. **«Tasa de error»**: `sum(rate(supportops_agent_requests_total{<SVC>,<TIER>,supportops_status=~"error|max_steps"}[$__rate_interval])) / sum(rate(…{<SVC>,<TIER>}[$__rate_interval]))`, unidad `percentunit`, umbrales **verde → naranja en 0,05 → rojo en 0,10**, y un **data link** «Ver logs de warn/error» a Explore (Loki): `{service_name="$service"} | detected_level=~"warn|error"`.
2. **«p95 del agente»**: `histogram_quantile(0.95, sum by (le) (rate(gen_ai_invoke_agent_duration_seconds_bucket{<SVC>,<TIER>}[$__rate_interval])))`, unidad `s`, umbrales **4** (naranja) y **6** (rojo), y un **data link** a Explore (Tempo) con las trazas `{ resource.service.name = "$service" && name =~ "invoke_agent.*" && duration > 4s }`.

(`SVC` y `TIER` ya están definidos arriba.) Cada panel necesita `description` (qué mide y cómo leerlo).

<details>
<summary>🪜 Pista 1</summary>

`dk.stat(titulo, expr, unit=..., thr=dk.thresholds((None, dk.GREEN), (0.05, dk.ORANGE), (0.10, dk.RED)), data_links=[dk.link(titulo, url)], description=...)`. La URL: `dk.explore_url("loki", {"expr": ..., "queryType": "range"})`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Consulta de error completa: `f'sum(rate(supportops_agent_requests_total{{{SVC},{TIER},supportops_status=~"error|max_steps"}}[{RATE}])) / sum(rate(supportops_agent_requests_total{{{SVC},{TIER}}}[{RATE}]))'`.

</details>

In [ ]:
def symptom_row(d: dk.Dashboard) -> dk.Dashboard:
    d.row("1 · ¿Estoy bien? Síntomas")
    # TODO 2: d.add(stat de error, stat de p95)
    raise NotImplementedError("TODO 2")

## Paso 3 · Tendencia (TODO 3)

Añade tres paneles:
1. **«Peticiones/s por estado»**: `timeseries` **apilada** (`stack=True`) con `sum by (supportops_status) (rate(supportops_agent_requests_total{<SVC>,<TIER>}[$__rate_interval]))`, leyenda `{{supportops_status}}`.
2. **«Latencia p50 / p95 / p99»**: un `timeseries` con **tres** consultas (`ref` A, B, C) `histogram_quantile(q, sum by (le)(rate(gen_ai_invoke_agent_duration_seconds_bucket{...}[$__rate_interval])))`, unidad `s`.
3. **«Distribución de la latencia»**: `dk.heatmap(...)` sobre `sum by (le) (rate(gen_ai_invoke_agent_duration_seconds_bucket{...}[$__rate_interval]))` con **`exemplar=True`** (los rombos que saltan a la traza).

<details>
<summary>🪜 Pista 1</summary>

`dk.timeseries(titulo, [dk.prom(expr, leyenda, ref="A"), ...], unit="s", description=...)`; `dk.heatmap(titulo, expr, exemplar=True, description=...)`. Recuerda: los percentiles **no** se apilan.

</details>

In [ ]:
def trend_row(d: dk.Dashboard) -> dk.Dashboard:
    # TODO 3: d.add(serie apilada por estado, serie con p50/p95/p99, mapa de calor con exemplars)
    raise NotImplementedError("TODO 3")

## Paso 4 · ¿Dónde se va el tiempo? (TODO 4)

Fila `2 · ¿Dónde se va el tiempo?` con tres `timeseries` (`w=8`) filtrando por `{SVC}`:
- **«p95 de la llamada al LLM»**: `histogram_quantile(0.95, sum by (le, gen_ai_request_model) (rate(gen_ai_client_inference_duration_seconds_bucket{...}[…])))`, leyenda `{{gen_ai_request_model}}`.
- **«Latencia media por herramienta»**: `sum by (gen_ai_tool_name) (rate(gen_ai_execute_tool_duration_seconds_sum{...}[…])) / sum by (gen_ai_tool_name) (rate(gen_ai_execute_tool_duration_seconds_count{...}[…]))`.
- **«Pasos por ticket (p90)»**: dos consultas, `histogram_quantile(0.9, sum by (le) (rate(gen_ai_invoke_agent_tool_calls_bucket{...}[…])))` y la de `gen_ai_invoke_agent_inference_calls_bucket`.

In [ ]:
def where_row(d: dk.Dashboard) -> dk.Dashboard:
    d.row("2 · ¿Dónde se va el tiempo?")
    # TODO 4
    raise NotImplementedError("TODO 4")

## Paso 5 · ¿Qué petición concreta? (TODO 5)

Fila `3 · ¿Por qué?`:
- **Tabla de trazas** (`dk.traces_table`, `w=12`): TraceQL `{ resource.service.name = "$service" && name =~ "invoke_agent.*" && duration > 4s }`. Añade un **override sobre el campo `traceID`** (`dk.field_link("traceID", enlace)`) cuyo enlace abra Explore (Tempo) con la consulta `${__value.raw}` (el id de la celda).
- **Panel de logs** (`dk.logs`, `w=12`): `{service_name="$service"} | detected_level=~"warn|error" |= "$search"`.

Para añadir el override a la tabla modifica `d.d["panels"][-1]["fieldConfig"]["overrides"]` justo después de `d.add(...)`.

<details>
<summary>🪜 Pista 1</summary>

`link = dk.link("Abrir la traza", dk.explore_url("tempo", {"queryType": "traceql", "query": "${__value.raw}"}))`; luego `d.d["panels"][-1]["fieldConfig"]["overrides"] = [dk.field_link("traceID", link)]` (el último panel añadido es la tabla).

</details>

In [ ]:
def why_row(d: dk.Dashboard) -> dk.Dashboard:
    d.row("3 · ¿Por qué? Trazas y logs concretos")
    # TODO 5
    raise NotImplementedError("TODO 5")

## Paso 6 · Enlaces de dashboard (TODO 6)

`finish(d)` añade **dos** enlaces de dashboard con `d.dash_link(...)`: uno **por *tags*** al dashboard de métricas GenAI (`tags=["metricas"]`) y otro a **Explore** (`url="/explore"`, `keep_time=True`, `include_vars=False`).

In [ ]:
def finish(d: dk.Dashboard) -> dk.Dashboard:
    # TODO 6
    raise NotImplementedError("TODO 6")

### El ensamblaje (te lo damos hecho)
Cada fila se aplica con tolerancia a fallos para que puedas ir viendo tu progreso:

In [ ]:
def build_dashboard() -> dk.Dashboard:
    d = dk.Dashboard(UID, "TEST: Salud del agente (alumno)", tags=["curso", "salud-agente", "supportops", "proj10"], time_from="now-30m",
                     description="Salud del agente de SupportOps: síntomas → dónde → por qué, con navegación entre señales.")
    try:
        for v in make_variables(): d.variable(v)
    except NotImplementedError: print("· variables pendientes")
    for fn in (symptom_row, trend_row, where_row, why_row, finish):
        try: fn(d)
        except NotImplementedError: print(f"· {fn.__name__} pendiente")
    return d

D = build_dashboard().to_dict()
print(len(panels(D)), "paneles ·", len(D["templating"]["list"]), "variables ·", len(D["links"]), "enlaces de dashboard")

## Paso 7 · Una alerta de síntoma (TODO 7)

Escribe `ALERT_YAML` (cadena YAML en el formato de *provisioning*) con **una regla** de tasa de error en un grupo evaluado cada `10s` (el intervalo de evaluación debe ser **múltiplo de 10 s**):
- consulta **A** a Prometheus (`datasourceUid: prometheus`, `instant: true`) con la **tasa de error** de `service_name="supportops-agent"` y ventana **explícita** `[2m]` (sin `$__rate_interval`);
- **B**: `reduce` (`reducer: last`) de A; **C**: `threshold` de B con `gt` y un umbral entre 0,05 y 0,20 (usa **0.10**); `condition: C`;
- `for` ≥ `30s`; `noDataState`; `execErrState: Error`;
- `labels` con `severity` (`page` o `ticket`); `annotations` con `summary` y `runbook_url`;
- en `annotations`, `__dashboardUid__` = el de **tu** dashboard (`UID`) y `__panelId__` = el **id** (como texto) de tu panel «Tasa de error».

<details>
<summary>🪜 Pista 1</summary>

Estructura: `apiVersion: 1` / `groups: [{orgId: 1, name: ..., folder: ..., interval: 15s, rules: [{uid, title, condition: C, for, noDataState, execErrState, data: [A, B, C], annotations (con __dashboardUid__ y __panelId__), labels}]}]`. Las expresiones B y C usan `datasourceUid: __expr__`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Mira `stack/grafana/provisioning/alerting/10-alert-rules.yaml` (la regla `sp-error-ratio-high`): copia la forma y **cambia** `uid`, `__dashboardUid__`, `__panelId__` y el `runbook_url`. El `panelId` sale de `titled(D, 'error')[0]['id']`.

</details>

In [ ]:
ALERT_YAML = """
# TODO 7: tu regla de alerta (YAML de provisioning)
"""

## Paso 8 · Rúbrica

In [ ]:
def run_rubric():
    rows = []
    def add(r, d, ok, det=""): rows.append({"criterio": r, "descripción": d, "resultado": ("✅ OK" if ok else "❌ FALLA") if ok is not None else "⏭ n/a", "detalle": det})
    D = None
    try: D = _quiet_build()
    except Exception as e: add("R0", "build_dashboard()", False, type(e).__name__); return pd.DataFrame(rows).set_index("criterio")
    V = {v["name"]: v for v in D["templating"]["list"]}
    # R1
    try:
        ok = (V["service"]["type"] == "query" and "label_values(" in str(V["service"]["query"]) and "service_name" in str(V["service"]["query"]) and not V["service"]["multi"] and not V["service"]["includeAll"]
              and V["tier"]["type"] == "query" and V["tier"]["multi"] and V["tier"]["includeAll"] and V["tier"].get("allValue") == ".*" and "supportops_customer_tier" in str(V["tier"]["query"])
              and V["search"]["type"] == "textbox")
        add("R1", "Variables service / tier / search", ok, f"{sorted(V)}")
    except Exception as e: add("R1", "Variables", False, type(e).__name__)
    # R2
    try:
        e_ = titled(D, "error")[0]; p_ = titled(D, "p95 del agente|p95 agente")[0]
        s_e = steps(e_); s_p = steps(p_); ex = exprs(e_); ep = exprs(p_)
        ok_e = (e_["type"] == "stat" and e_["fieldConfig"]["defaults"]["unit"] == "percentunit" and 0.05 in s_e and 0.1 in s_e and "$service" in ex and "$tier" in ex and RATE in ex
                and "error|max_steps" in ex.replace(" ", "") and any("/explore" in l["url"] and "loki" in l["url"] for l in data_links(e_)))
        ok_p = (p_["type"] == "stat" and "histogram_quantile(0.95" in ep.replace(" ", "") and "sum by (le)" in ep and RATE in ep and 4 in s_p and 6 in s_p and "$service" in ep
                and any("/explore" in l["url"] and "tempo" in l["url"] for l in data_links(p_)))
        add("R2", "Stats de error y p95: consulta, unidad, umbrales y data links", ok_e and ok_p, f"error={ok_e} · p95={ok_p}")
    except Exception as e: add("R2", "Síntomas", False, type(e).__name__ + ": " + str(e)[:50])
    # R3
    try:
        st = [p for p in titled(D, "estado") if p["type"] == "timeseries"][0]; pc = [p for p in panels(D) if p["type"] == "timeseries" and len(p["targets"]) == 3 and "histogram_quantile" in exprs(p)][0]
        hm = [p for p in panels(D) if p["type"] == "heatmap"][0]
        qs = " ".join(t["expr"].replace(" ", "") for t in pc["targets"])
        ok = (st["fieldConfig"]["defaults"]["custom"]["stacking"]["mode"] == "normal" and "sumby(supportops_status)" in exprs(st).replace(" ", "")
              and all(f"histogram_quantile({q}" in qs for q in ("0.5", "0.95", "0.99")) and any(t.get("exemplar") is True for t in hm["targets"]) and hm["targets"][0].get("format") == "heatmap" and "_bucket" in exprs(hm))
        add("R3", "Serie apilada por estado, 3 percentiles y mapa de calor con exemplars", ok)
    except Exception as e: add("R3", "Tendencia", False, type(e).__name__ + ": " + str(e)[:50])
    # R4
    try:
        a = titled(D, "LLM")[0]; b = titled(D, "herramienta")[0]; c = titled(D, "pasos")[0]
        ok = ("gen_ai_client_inference_duration_seconds_bucket" in exprs(a) and "by (le, gen_ai_request_model)" in exprs(a) and "gen_ai_execute_tool_duration_seconds_sum" in exprs(b) and "gen_ai_execute_tool_duration_seconds_count" in exprs(b)
              and "gen_ai_tool_name" in exprs(b) and "gen_ai_invoke_agent_tool_calls_bucket" in exprs(c) and "gen_ai_invoke_agent_inference_calls_bucket" in exprs(c) and "0.9" in exprs(c) and all("$service" in exprs(p) for p in (a, b, c)))
        add("R4", "Fila «dónde»: LLM por modelo, herramientas (sum/count) y pasos p90", ok)
    except Exception as e: add("R4", "Dónde", False, type(e).__name__ + ": " + str(e)[:50])
    # R5
    try:
        tb = [p for p in panels(D) if p["type"] == "table" and (p.get("datasource") or {}).get("uid") == "tempo"][0]; lg = [p for p in panels(D) if p["type"] == "logs"][0]
        ov = [o for o in tb["fieldConfig"]["overrides"] if o["matcher"]["options"] == "traceID"]
        okt = tb["targets"][0].get("queryType") == "traceql" and "$service" in exprs(tb) and "duration" in exprs(tb) and bool(ov) and "${__value.raw}" in json.dumps(ov[0])
        okl = (lg.get("datasource") or {}).get("uid") == "loki" and "$service" in exprs(lg) and "$search" in exprs(lg) and "detected_level" in exprs(lg)
        add("R5", "Tabla de trazas con enlace en traceID y panel de logs", okt and okl, f"trazas={okt} · logs={okl}")
    except Exception as e: add("R5", "Por qué", False, type(e).__name__ + ": " + str(e)[:50])
    # R6
    try:
        n_dl = sum(len(data_links(p)) for p in panels(D)); dl = D["links"]
        ok = n_dl >= 3 and len(dl) >= 2 and any(l.get("type") == "dashboards" and "metricas" in l.get("tags", []) for l in dl)
        add("R6", "Navegación: ≥ 3 data links y ≥ 2 enlaces de dashboard (uno por tags)", ok, f"data links={n_dl} · dashboard links={len(dl)}")
    except Exception as e: add("R6", "Navegación", False, type(e).__name__)
    # R7
    try:
        problems = dk.lint(D); h1 = hashlib.sha256(_quiet_build_json().encode()).hexdigest(); h2 = hashlib.sha256(_quiet_build_json().encode()).hexdigest()
        add("R7", "Higiene (lint sin problemas, ≥ 10 paneles) y determinismo", not problems and h1 == h2 and len(panels(D)) >= 10, "; ".join(problems[:3]) or f"{len(panels(D))} paneles · hash {h1[:10]}")
    except Exception as e: add("R7", "Higiene", False, type(e).__name__)
    # R8
    try:
        y = yaml.safe_load(ALERT_YAML); r = y["groups"][0]["rules"][0]; data = {x["refId"]: x for x in r["data"]}
        A, B, C = data["A"], data["B"], data["C"]
        thr = C["model"]["conditions"][0]["evaluator"]; win = re.findall(r"\[(\d+)([smh])\]", A["model"]["expr"]); secs = lambda s: int(re.match(r"\d+", str(s)).group()) * {"s": 1, "m": 60, "h": 3600}[re.search(r"[smh]", str(s)).group()]
        pid = titled(D, "error")[0]["id"]
        checks = {"apiVersion": y["apiVersion"] == 1, "grupo múltiplo de 10s": str(y["groups"][0]["interval"]) in ("10s", "20s", "30s", "1m"), "A prometheus instant": A["datasourceUid"] == "prometheus" and A["model"].get("instant") is True,
                  "A: tasa de error": "supportops_agent_requests_total" in A["model"]["expr"] and "error" in A["model"]["expr"] and "/" in A["model"]["expr"],
                  "ventana ≥ 1m explícita": bool(win) and all(int(n) * {"s": 1, "m": 60, "h": 3600}[u] >= 60 for n, u in win) and "__rate_interval" not in A["model"]["expr"],
                  "B reduce last": B["model"]["type"] == "reduce" and B["model"]["reducer"] == "last" and B["model"]["expression"] == "A",
                  "C umbral gt 0,05-0,20": C["model"]["type"] == "threshold" and thr["type"] == "gt" and 0.05 <= float(thr["params"][0]) <= 0.20 and C["model"]["expression"] == "B" and r["condition"] == "C",
                  "for ≥ 30s": secs(r["for"]) >= 30, "etiquetas/anotaciones": r["labels"].get("severity") in ("page", "ticket") and r["annotations"].get("summary") and r["annotations"].get("runbook_url"),
                  "enlace a tu dashboard y panel": r["annotations"].get("__dashboardUid__") == UID and int(r["annotations"].get("__panelId__", -1)) == pid, "execErrState": r.get("execErrState") in ("Error", "Alerting", "OK"), "noDataState": r.get("noDataState") in ("OK", "NoData", "Alerting")}
        bad = [k for k, v in checks.items() if not v]
        add("R8", "Regla de alerta aprovisionable (11 comprobaciones)", not bad, "todas OK" if not bad else "fallan: " + ", ".join(bad))
    except Exception as e: add("R8", "Alerta", False, type(e).__name__ + ": " + str(e)[:60])
    # L1-L2 (informativas) -------------------------------------------------------------------------------
    if not LIVE:
        add("L1", "Consultas ejecutan contra Grafana (con stack)", None, "sin stack"); add("L2", "El dashboard se sube y se lee (con stack)", None, "sin stack")
    else:
        try:
            chk = dk.query_check(D, overrides={"service": "supportops-agent"}); bad = [(c["panel"], c["error"][:60]) for c in chk if not c["ok"]]
            add("L1", "Todas las consultas del dashboard ejecutan sin error en Grafana", not bad, f"{len(chk) - len(bad)}/{len(chk)} OK " + (str(bad[:2]) if bad else ""))
            dk.upload(D); g = dk.grafana("GET", f"/api/dashboards/uid/{UID}")
            add("L2", "El dashboard se sube por la API y se lee de vuelta", g["dashboard"]["uid"] == UID, g["meta"]["url"])
        except Exception as e: add("L1", "(en vivo)", False, type(e).__name__ + ": " + str(e)[:70])
    return pd.DataFrame(rows).set_index("criterio")

def _quiet_build():
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()): return build_dashboard().to_dict()
def _quiet_build_json():
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()): return build_dashboard().to_json()

In [ ]:
# Ejecuta esta celda cuando hayas terminado los TODO
display(run_rubric())

---
## Retos extra (nivel experto)

1. **Datasource como variable:** convierte `prometheus` en una variable `$ds` de tipo *datasource* y usa `${ds}` en los paneles para reutilizar el dashboard con otro Prometheus (¿qué cambia en `query_check`?).
2. **Repetición:** haz que el panel «Latencia media por herramienta» se **repita por herramienta** (`repeat` sobre una variable multi-valor `tool`) en vez de dibujar todas en una gráfica. ¿Cuándo es mejor una cosa u otra?
3. **Alerta con enlace profundo:** añade a la anotación `description` una URL a tu dashboard con `?var-service=...&from=...&to=...` usando las variables de plantilla de Grafana (`{{ $labels.service }}`, `{{ $values }}`).
4. **CI del dashboard:** escribe un script `check_dashboards.py` que recorra `stack/grafana/dashboards/*.json`, ejecute `lint()` y falle si hay problemas; añade una segunda comprobación que verifique que **cada alerta** apunta a un `dashboardUid` que existe.
5. **Segunda alerta:** crea la regla de **latencia** (p95 > 6 s, `severity=ticket`) y una **política** que la enrute con `group_wait` mayor que la de `page`.

## Reflexión (producción e hilo conductor)

- Tu dashboard responde a *«¿estoy bien?»* en 5 segundos para quien lo mira. ¿Y para quien llega por una alerta de **latencia** a las 3 de la mañana? ¿Qué cambiarías en el orden de las filas?
- ¿Qué señal necesita un panel que **no** tienes? (calidad de las respuestas, módulo 14; coste por cliente, módulo 13; SLO y *burn rate*, módulo 16).
- ¿Cuántas series y cuántas consultas por minuto genera tu dashboard con 20 usuarios mirándolo cada 10 s? ¿Qué *recording rules* añadirías?
- En el capstone (22) un solo dashboard no basta: ¿cómo organizarías **carpetas, permisos y enlaces** entre dashboards de agente, backend y coste?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable (redefine los TODO, sube el dashboard y vuelve a pasar la rúbrica).

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
# TODO 1
def make_variables() -> list:
    return [dk.var_query("service", "label_values(supportops_agent_requests_total, service_name)", label="Servicio", multi=False, all_=False, current="supportops-agent"),
            dk.var_query("tier", "label_values(supportops_agent_requests_total, supportops_customer_tier)", label="Tier"),
            dk.var_textbox("search", label="Buscar en logs")]

# TODO 2
def symptom_row(d: dk.Dashboard) -> dk.Dashboard:
    d.row("1 · ¿Estoy bien? Síntomas")
    err_logs = dk.explore_url("loki", {"expr": '{service_name="$service"} | detected_level=~"warn|error"', "queryType": "range"})
    slow = dk.explore_url("tempo", {"queryType": "traceql", "query": '{ resource.service.name = "$service" && name =~ "invoke_agent.*" && duration > 4s }', "limit": 20, "tableType": "traces"})
    d.add(
        dk.stat("Tasa de error", f'sum(rate(supportops_agent_requests_total{{{SVC},{TIER},supportops_status=~"error|max_steps"}}[{RATE}])) / sum(rate(supportops_agent_requests_total{{{SVC},{TIER}}}[{RATE}]))',
                unit="percentunit", decimals=1, thr=dk.thresholds((None, dk.GREEN), (0.05, dk.ORANGE), (0.10, dk.RED)), w=4,
                description="Fracción de tickets en error o max_steps. Naranja > 5 %, rojo > 10 % (umbral de la alerta). Enlace: logs de warn/error.", data_links=[dk.link("Ver logs de warn/error", err_logs)]),
        dk.stat("p95 del agente", f'histogram_quantile(0.95, sum by (le) (rate(gen_ai_invoke_agent_duration_seconds_bucket{{{SVC},{TIER}}}[{RATE}])))', unit="s", decimals=2,
                thr=dk.thresholds((None, dk.GREEN), (4, dk.ORANGE), (6, dk.RED)), w=4,
                description="p95 de invoke_agent por interpolación de buckets. Naranja > 4 s, rojo > 6 s. Enlace: trazas lentas.", data_links=[dk.link("Ver trazas lentas (> 4 s)", slow)]),
    )
    return d

# TODO 3
def trend_row(d: dk.Dashboard) -> dk.Dashboard:
    d.add(
        dk.timeseries("Peticiones/s por estado", [dk.prom(f'sum by (supportops_status) (rate(supportops_agent_requests_total{{{SVC},{TIER}}}[{RATE}]))', "{{supportops_status}}")], unit="reqps", stack=True, w=8,
                      description="Tasa por estado final (excluyentes: por eso se puede apilar)."),
        dk.timeseries("Latencia p50 / p95 / p99", [dk.prom(f'histogram_quantile({q}, sum by (le) (rate(gen_ai_invoke_agent_duration_seconds_bucket{{{SVC},{TIER}}}[{RATE}])))', f"p{int(q * 100)}", ref=r)
                                                    for q, r in ((0.5, "A"), (0.95, "B"), (0.99, "C"))], unit="s", w=8, description="Percentiles por interpolación de buckets (no se apilan)."),
        dk.heatmap("Distribución de la latencia (exemplars)", f'sum by (le) (rate(gen_ai_invoke_agent_duration_seconds_bucket{{{SVC},{TIER}}}[{RATE}]))', w=8, h=8, exemplar=True,
                   description="Mapa de calor de buckets; los rombos son exemplars que abren la traza en Tempo."),
    )
    return d

# TODO 4
def where_row(d: dk.Dashboard) -> dk.Dashboard:
    d.row("2 · ¿Dónde se va el tiempo?")
    d.add(
        dk.timeseries("p95 de la llamada al LLM", [dk.prom(f'histogram_quantile(0.95, sum by (le, gen_ai_request_model) (rate(gen_ai_client_inference_duration_seconds_bucket{{{SVC}}}[{RATE}])))', "{{gen_ai_request_model}}")],
                      unit="s", w=8, description="Latencia del proveedor LLM por modelo."),
        dk.timeseries("Latencia media por herramienta", [dk.prom(f'sum by (gen_ai_tool_name) (rate(gen_ai_execute_tool_duration_seconds_sum{{{SVC}}}[{RATE}])) / sum by (gen_ai_tool_name) (rate(gen_ai_execute_tool_duration_seconds_count{{{SVC}}}[{RATE}]))', "{{gen_ai_tool_name}}")],
                      unit="s", w=8, description="sum/count del histograma por herramienta."),
        dk.timeseries("Pasos por ticket (p90)", [dk.prom(f'histogram_quantile(0.9, sum by (le) (rate(gen_ai_invoke_agent_tool_calls_bucket{{{SVC}}}[{RATE}])))', "herramientas", ref="A"),
                                                  dk.prom(f'histogram_quantile(0.9, sum by (le) (rate(gen_ai_invoke_agent_inference_calls_bucket{{{SVC}}}[{RATE}])))', "llamadas LLM", ref="B")],
                      unit="short", w=8, description="Un p90 que sube delata bucles o reintentos."),
    )
    return d

# TODO 5
def why_row(d: dk.Dashboard) -> dk.Dashboard:
    d.row("3 · ¿Por qué? Trazas y logs concretos")
    link = dk.link("Abrir la traza", dk.explore_url("tempo", {"queryType": "traceql", "query": "${__value.raw}"}))
    d.add(dk.traces_table("Trazas más lentas (> 4 s)", '{ resource.service.name = "$service" && name =~ "invoke_agent.*" && duration > 4s }', w=12, h=9,
                          description="TraceQL: invoke_agent de más de 4 s. Clic en el traceID: menú de enlaces con «Abrir la traza»."))
    d.d["panels"][-1]["fieldConfig"]["overrides"] = [dk.field_link("traceID", link)]
    d.add(dk.logs("Logs de warn y error", '{service_name="$service"} | detected_level=~"warn|error" |= "$search"', w=12, h=9,
                  description="LogQL: logs warn/error del servicio; filtra con la variable «Buscar en logs»."))
    return d

# TODO 6
def finish(d: dk.Dashboard) -> dk.Dashboard:
    d.dash_link("Métricas GenAI (módulo 09)", "", tags=["metricas"], as_dropdown=False)
    d.dash_link("Explore", "/explore", keep_time=True, include_vars=False, icon="search")
    return d

# TODO 7
ALERT_YAML = f"""
apiVersion: 1
groups:
  - orgId: 1
    name: proj10-sintomas
    folder: Curso observabilidad de agentes
    interval: 10s
    rules:
      - uid: proj10-error-ratio
        title: "PROJ10 · tasa de error alta"
        condition: C
        for: 30s
        noDataState: OK
        execErrState: Error
        data:
          - refId: A
            relativeTimeRange: {{from: 300, to: 0}}
            datasourceUid: prometheus
            model:
              refId: A
              expr: 'sum(rate(supportops_agent_requests_total{{service_name="supportops-agent",supportops_status=~"error|max_steps"}}[2m])) / sum(rate(supportops_agent_requests_total{{service_name="supportops-agent"}}[2m]))'
              instant: true
          - refId: B
            datasourceUid: __expr__
            model: {{refId: B, type: reduce, expression: A, reducer: last, settings: {{mode: dropNN}}}}
          - refId: C
            datasourceUid: __expr__
            model:
              refId: C
              type: threshold
              expression: B
              conditions:
                - evaluator: {{type: gt, params: [0.10]}}
        annotations:
          __dashboardUid__: {UID}
          __panelId__: "{titled(_quiet_build(), 'error')[0]['id']}"
          summary: "Más del 10 % de los tickets terminan en error"
          runbook_url: "https://example.com/runbooks/proj10-error-ratio"
        labels:
          severity: page
          team: plataforma
"""
print(len(panels(_quiet_build())), "paneles; regla de alerta enlazada al panel", yaml.safe_load(ALERT_YAML)["groups"][0]["rules"][0]["annotations"]["__panelId__"])
display(run_rubric())
if LIVE:
    dk.grafana("DELETE", f"/api/dashboards/uid/{UID}"); print("dashboard del alumno borrado del Grafana de laboratorio")

In [ ]:
# Visualización de la solución: mapa del dashboard del alumno (gridPos) y número de consultas por panel
Dsol = _quiet_build(); fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1.3, 1]})
kinds = {"stat": viz.PALETTE["metrics"], "timeseries": viz.PALETTE["traces"], "heatmap": viz.PALETTE["profiles"], "table": viz.PALETTE["neutral"], "logs": viz.PALETTE["logs"], "row": "#e5e7eb"}
ymax = 0
for p in Dsol["panels"]:
    g = p["gridPos"]; ymax = max(ymax, g["y"] + g["h"]); c = kinds.get(p["type"], "#999")
    axes[0].add_patch(plt.Rectangle((g["x"] + .08, g["y"] + .08), g["w"] - .16, g["h"] - .16, facecolor=c + "55" if p["type"] != "row" else c, edgecolor=c))
    axes[0].text(g["x"] + g["w"] / 2, g["y"] + g["h"] / 2, p["title"][:24], ha="center", va="center", fontsize=6.5)
axes[0].set_xlim(0, 24); axes[0].set_ylim(ymax, 0); axes[0].set_title("Mapa de tu dashboard (gridPos)"); axes[0].grid(False)
names = [p["title"][:26] for p in panels(Dsol)]; nq = [len(p["targets"]) for p in panels(Dsol)]
axes[1].barh(names, nq, color=viz.PALETTE["traces"]); axes[1].invert_yaxis(); axes[1].set_xlabel("consultas (targets)"); axes[1].set_title("Consultas por panel")
plt.tight_layout(); plt.show()